# Teste de Extração: IBAMA Embargos

Baixa o Shapefile de embargos do IBAMA e converte para Parquet (geoparquet + tabular).

**Requisitos:** `pip install geopandas requests pyogrio`

## URLs Oficiais (março/2026)

| Tipo | URL |
|------|-----|
| **Embargos (principal)** | https://pamgia.ibama.gov.br/geoservicos/arquivos/adm_embargo_ibama_a.shp.zip |
| **Autos de Infração** | https://pamgia.ibama.gov.br/geoservicos/arquivos/adm_auto_infracao_ibama_a.shp.zip |
| **WFS Service** | https://pamgia.ibama.gov.br/server/services/01_Publicacoes_Bases/adm_embargos_ibama_a/MapServer/WFSServer |
| **Portal Dados Abertos** | https://dadosabertos.ibama.gov.br/dataset/termos-de-embargo |

## Características do Dataset

- **Nome fixo:** `adm_embargo_ibama_a.shp.zip` (singular "embargo")
- **Atualização:** IBAMA substitui o arquivo periodicamente (não há versão por ano/mês)
- **Encoding:** `iso-8859-1` (latin1) para atributos textuais
- **Formato:** Shapefile compactado em ZIP

## Troubleshooting

| Problema | Solução |
|----------|----------|
| Erro 503/403 | User-Agent + retry com backoff |
| "Nenhum .shp encontrado" | Inspecionar ZIP antes de ler |
| Erro de encoding | Usar `encoding="iso-8859-1"` |
| Falha na leitura | Especificar camada exata + engine pyogrio |

In [1]:
import requests
import geopandas as gpd
import pandas as pd
import zipfile
import tempfile
import os
from pathlib import Path
from time import sleep

URL = "https://pamgia.ibama.gov.br/geoservicos/arquivos/adm_embargo_ibama_a.shp.zip"
OUTPUT = Path("../../../data/01_bronze/ibama/ibama_embargos")
OUTPUT.mkdir(parents=True, exist_ok=True)

def baixar_com_retry(url, max_tentativas=5):
    """Baixa arquivo com retry automático e backoff para lidar com 503/403."""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    for tentativa in range(max_tentativas):
        try:
            r = requests.get(url, stream=True, verify=False, headers=headers, timeout=60)
            r.raise_for_status()
            return r
        except Exception as e:
            print(f"Tentativa {tentativa+1}/{max_tentativas} falhou: {e}")
            sleep(3 * (tentativa + 1))  # backoff exponencial
    raise Exception("Falha após todas as tentativas")

print("Baixando Áreas Embargadas IBAMA (pode demorar)...")
try:
    r = baixar_com_retry(URL)
    
    with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp:
        for chunk in r.iter_content(chunk_size=8192 * 4):
            tmp.write(chunk)
        tmp_path = tmp.name

    print(f"Download concluído ({os.path.getsize(tmp_path) / (1024*1024):.1f} MB)")

    # === INSPEÇÃO DO ZIP (importantíssimo) ===
    with zipfile.ZipFile(tmp_path) as z:
        shapefiles = [n for n in z.namelist() if n.lower().endswith('.shp')]
        print(f"Shapefiles encontrados no ZIP: {shapefiles}")
        shp_layer = shapefiles[0] if shapefiles else None

    if not shp_layer:
        raise Exception("Nenhum .shp encontrado dentro do ZIP!")

    # === LEITURA COM PARÂMETROS CORRETOS ===
    print(f"Lendo camada: {shp_layer} ...")
    gdf = gpd.read_file(
        f"zip://{tmp_path}!{shp_layer}",
        encoding="iso-8859-1",      # ← ESSA É A CHAVE
        engine="pyogrio"            # mais rápido e moderno
    )

    print(f"✅ Sucesso! {len(gdf):,} áreas embargadas carregadas.")
    print(f"Colunas: {list(gdf.columns)}")

    # Salva versão completa (geoparquet) + versão tabular
    gdf.to_parquet(OUTPUT / "embargos_ibama_full.geoparquet", index=False)
    
    df_tabular = gdf.drop(columns=['geometry'])
    df_tabular.to_parquet(OUTPUT / "embargos_ibama_tabular.parquet", index=False)

    os.remove(tmp_path)
    print(f"Arquivos salvos em {OUTPUT}")

except Exception as e:
    print(f"❌ Falha: {e}")
    if 'tmp_path' in locals() and os.path.exists(tmp_path):
        print(f"ZIP parcial salvo em: {tmp_path} (para debug)")

Baixando Áreas Embargadas IBAMA (pode demorar)...


/home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'pamgia.ibama.gov.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Download concluído (61.4 MB)
Shapefiles encontrados no ZIP: ['adm_embargo_ibama_a.shp']
Lendo camada: adm_embargo_ibama_a.shp ...
✅ Sucesso! 88,586 áreas embargadas carregadas.
Colunas: ['objectid', 'seq_tad', 'num_tad', 'serie_tad', 'operacao', 'origem_geo', 'cod_uf', 'uf', 'cod_munici', 'municipio', 'nome_imove', 'des_locali', 'nome_embar', 'cpf_cnpj_e', 'sit_desmat', 'tipo_area', 'num_auto_i', 'serie_auto', 'cod_tipo_b', 'des_tipo_b', 'unid_contr', 'ordem_fisc', 'cd_acao_fi', 'num_proces', 'des_tad', 'des_infrac', 'num_longit', 'num_latitu', 'dat_embarg', 'dat_impres', 'dat_ult_al', 'num_long00', 'num_lati00', 'qtd_area_d', 'qtd_area_e', 'dat_ult_00', 'st_area(sh', 'st_perimet', 'geometry']
Arquivos salvos em data/bronze/ibama_embargos
